# 01 — Fairness dataset diagnostics

Este notebook executa um **diagnóstico exploratório por base** para apoiar a sugestão da orientadora sobre verificar a associação entre:

- atributo protegido e target;
- features e atributo protegido;
- features e target.

Ele **não faz parte do pipeline R30**. A ideia é rodá-lo uma vez por base e salvar artefatos úteis tanto para discussão metodológica quanto para escrita do artigo.

## O que este notebook gera

Para cada base:

- `dataset_summary.csv`
- `protected_target_contingency.csv`
- `protected_target_correlation.csv`
- `feature_protected_correlation.csv`
- `feature_target_correlation.csv`
- `top_feature_correlation_summary.csv`
- `top_features_vs_protected.png`
- `top_features_vs_target.png`
- `correlation_heatmap_top_features.png`

As saídas ficam em `artifacts/dataset_diagnostics/<dataset_name>/`.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

TOP_N = 20

ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent if Path.cwd().parent != Path.cwd() else Path.cwd(),
    Path("/mnt/data"),
]

OUTPUT_ROOT = Path.cwd() / "artifacts" / "dataset_diagnostics"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "bank_marketing": {
        "paths": [
            "data/source/bank/bank-additional-full.csv",
            "data/source/fairness/bank-additional-full.csv",
            "bank-additional-full.csv",
            "/mnt/data/bank-additional-full.csv",
        ],
        "loader": "bank",
        "protected_label": "age_ge_25",
        "direct_protected_source_cols": ["age"],
        "target_label": "y_yes_is_1",
    },
    "credit_card_default": {
        "paths": [
            "data/source/fairness/credit_card_default.csv",
            "data/source/credit_card_default.csv",
            "credit_card_default.csv",
            "/mnt/data/credit_card_default.csv",
        ],
        "loader": "credit_card",
        "protected_label": "sex_male_is_1",
        "direct_protected_source_cols": ["SEX"],
        "target_label": "y_positive_is_1",
    },
    "german_credit": {
        "paths": [
            "data/source/fairness/german_credit.csv",
            "german_credit.csv",
            "/mnt/data/german_credit.csv",
        ],
        "loader": "german",
        "protected_label": "age_ge_25",
        "direct_protected_source_cols": ["age"],
        "target_label": "y_positive_is_1",
    },
}

def resolve_path(candidates):
    for candidate in candidates:
        p = Path(candidate)
        if p.is_absolute() and p.exists():
            return p
        for root in ROOT_CANDIDATES:
            test = (root / candidate).resolve()
            if test.exists():
                return test
    raise FileNotFoundError(f"Arquivo não encontrado. Tentativas: {candidates}")

def load_bank(path: Path):
    df = pd.read_csv(path, sep=";")
    obj_cols = df.select_dtypes(include=["object"]).columns.tolist()
    if obj_cols:
        mask_unknown = pd.Series(False, index=df.index)
        for col in obj_cols:
            mask_unknown = mask_unknown | (df[col].astype(str).str.lower() == "unknown")
        df = df.loc[~mask_unknown].copy()

    y = (df["y"].astype(str).str.lower() == "yes").astype(int)
    protected = (pd.to_numeric(df["age"], errors="coerce") >= 25).astype(int)
    X = df.drop(columns=["y"]).copy()
    return X, y, protected

def load_credit_card(path: Path):
    df = pd.read_csv(path)
    y = pd.to_numeric(df["y"], errors="coerce").fillna(0).astype(int)
    protected = (pd.to_numeric(df["SEX"], errors="coerce") == 1).astype(int)
    X = df.drop(columns=["y"]).copy()
    return X, y, protected

def load_german(path: Path):
    df = pd.read_csv(path)
    y = pd.to_numeric(df["y"], errors="coerce").fillna(0).astype(int)
    protected = (pd.to_numeric(df["age"], errors="coerce") >= 25).astype(int)
    X = df.drop(columns=["y"]).copy()
    return X, y, protected

LOADERS = {
    "bank": load_bank,
    "credit_card": load_credit_card,
    "german": load_german,
}

def encode_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    bool_cols = X.select_dtypes(include=["bool"]).columns.tolist()
    for col in bool_cols:
        X[col] = X[col].astype(int)

    obj_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    X_encoded = pd.get_dummies(X, columns=obj_cols, drop_first=False, dtype=float)

    for col in X_encoded.columns:
        if X_encoded[col].dtype == bool:
            X_encoded[col] = X_encoded[col].astype(int)

    X_encoded = X_encoded.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    X_encoded.columns = [str(c) for c in X_encoded.columns]
    return X_encoded

def safe_corr(a: pd.Series, b: pd.Series) -> float:
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    if a.nunique(dropna=True) <= 1 or b.nunique(dropna=True) <= 1:
        return 0.0
    value = a.corr(b)
    if pd.isna(value):
        return 0.0
    return float(value)

def correlation_table(X_encoded: pd.DataFrame, binary_series: pd.Series, label: str) -> pd.DataFrame:
    rows = []
    for col in X_encoded.columns:
        rows.append({
            "feature": col,
            f"corr_with_{label}": safe_corr(X_encoded[col], binary_series),
        })
    out = pd.DataFrame(rows)
    out[f"abs_corr_with_{label}"] = out[f"corr_with_{label}"].abs()
    out = out.sort_values(f"abs_corr_with_{label}", ascending=False).reset_index(drop=True)
    return out

def dataset_summary_df(dataset_name: str, X: pd.DataFrame, y: pd.Series, protected: pd.Series, protected_label: str) -> pd.DataFrame:
    summary = {
        "dataset": dataset_name,
        "n_rows": int(len(X)),
        "n_features_raw": int(X.shape[1]),
        "target_positive_rate": float(y.mean()),
        f"{protected_label}_rate": float(protected.mean()),
        f"target_positive_rate_{protected_label}_0": float(y.loc[protected == 0].mean()) if (protected == 0).any() else np.nan,
        f"target_positive_rate_{protected_label}_1": float(y.loc[protected == 1].mean()) if (protected == 1).any() else np.nan,
        "n_target_positive": int(y.sum()),
        "n_target_negative": int((1 - y).sum()),
        f"n_{protected_label}_1": int(protected.sum()),
        f"n_{protected_label}_0": int((1 - protected).sum()),
    }
    return pd.DataFrame([summary])

def save_barplot(df: pd.DataFrame, value_col: str, title: str, output_path: Path):
    fig, ax = plt.subplots(figsize=(10, 6))
    plot_df = df.iloc[:TOP_N].iloc[::-1]
    ax.barh(plot_df["feature"], plot_df[value_col])
    ax.set_title(title)
    ax.set_xlabel("Correlação")
    ax.set_ylabel("Feature")
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)

def save_heatmap(corr_matrix: pd.DataFrame, title: str, output_path: Path):
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr_matrix.values, aspect="auto")
    ax.set_xticks(range(len(corr_matrix.columns)))
    ax.set_xticklabels(corr_matrix.columns, rotation=90)
    ax.set_yticks(range(len(corr_matrix.index)))
    ax.set_yticklabels(corr_matrix.index)
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)

def run_dataset_diagnostics(dataset_name: str, meta: dict):
    path = resolve_path(meta["paths"])
    X_raw, y, protected = LOADERS[meta["loader"]](path)
    X_encoded = encode_features(X_raw)

    out_dir = OUTPUT_ROOT / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    summary = dataset_summary_df(dataset_name, X_raw, y, protected, meta["protected_label"])
    summary.to_csv(out_dir / "dataset_summary.csv", index=False)

    contingency = pd.crosstab(
        pd.Series(protected, name=meta["protected_label"]),
        pd.Series(y, name=meta["target_label"]),
        dropna=False,
    )
    contingency.to_csv(out_dir / "protected_target_contingency.csv")

    protected_target_corr = safe_corr(pd.Series(protected), pd.Series(y))
    pd.DataFrame([{
        "dataset": dataset_name,
        "protected_label": meta["protected_label"],
        "target_label": meta["target_label"],
        "phi_or_pearson_binary_correlation": protected_target_corr,
    }]).to_csv(out_dir / "protected_target_correlation.csv", index=False)

    feature_protected = correlation_table(X_encoded, pd.Series(protected), meta["protected_label"])
    feature_target = correlation_table(X_encoded, pd.Series(y), meta["target_label"])

    feature_protected.to_csv(out_dir / "feature_protected_correlation.csv", index=False)
    feature_target.to_csv(out_dir / "feature_target_correlation.csv", index=False)

    exclusion = set(meta.get("direct_protected_source_cols", []))
    feature_protected_filtered = feature_protected.loc[~feature_protected["feature"].isin(exclusion)].reset_index(drop=True)

    top_union = list(dict.fromkeys(
        feature_protected_filtered["feature"].head(TOP_N).tolist()
        + feature_target["feature"].head(TOP_N).tolist()
    ))

    heatmap_df = X_encoded[top_union].copy()
    heatmap_df[meta["protected_label"]] = protected.values
    heatmap_df[meta["target_label"]] = y.values
    corr_matrix = heatmap_df.corr().fillna(0.0)

    top_summary = pd.DataFrame({
        "feature": top_union,
    }).merge(
        feature_protected[["feature", f"corr_with_{meta['protected_label']}", f"abs_corr_with_{meta['protected_label']}"]],
        on="feature",
        how="left",
    ).merge(
        feature_target[["feature", f"corr_with_{meta['target_label']}", f"abs_corr_with_{meta['target_label']}"]],
        on="feature",
        how="left",
    ).sort_values(
        [f"abs_corr_with_{meta['protected_label']}", f"abs_corr_with_{meta['target_label']}"],
        ascending=False,
    )
    top_summary.to_csv(out_dir / "top_feature_correlation_summary.csv", index=False)

    save_barplot(
        feature_protected_filtered,
        f"corr_with_{meta['protected_label']}",
        f"{dataset_name} — top features vs protected",
        out_dir / "top_features_vs_protected.png",
    )
    save_barplot(
        feature_target,
        f"corr_with_{meta['target_label']}",
        f"{dataset_name} — top features vs target",
        out_dir / "top_features_vs_target.png",
    )
    save_heatmap(
        corr_matrix,
        f"{dataset_name} — correlation heatmap (top features + protected + target)",
        out_dir / "correlation_heatmap_top_features.png",
    )

    return {
        "dataset": dataset_name,
        "path_used": str(path),
        "n_rows": int(len(X_raw)),
        "n_features_raw": int(X_raw.shape[1]),
        "n_features_encoded": int(X_encoded.shape[1]),
        "protected_target_corr": protected_target_corr,
        "output_dir": str(out_dir),
    }

In [2]:
results = []
for dataset_name, meta in DATASETS.items():
    result = run_dataset_diagnostics(dataset_name, meta)
    results.append(result)

overview = pd.DataFrame(results).sort_values("dataset").reset_index(drop=True)
overview.to_csv(OUTPUT_ROOT / "dataset_diagnostics_overview.csv", index=False)
overview

,dataset,path_used,n_rows,n_features_raw,n_features_encoded,protected_target_corr,output_dir
0,bank_marketing,/Users/caiotertuliano/Projects/doe_nbi_hpo_pro...,30488,19,56,-0.053289,/Users/caiotertuliano/Projects/doe_nbi_hpo_pro...
1,credit_card_default,/Users/caiotertuliano/Projects/doe_nbi_hpo_pro...,30000,23,23,-0.039961,/Users/caiotertuliano/Projects/doe_nbi_hpo_pro...
2,german_credit,/Users/caiotertuliano/Projects/doe_nbi_hpo_pro...,1000,20,61,0.099890,/Users/caiotertuliano/Projects/doe_nbi_hpo_pro...


## Como interpretar rapidamente

- **`protected_target_correlation.csv`**: mostra se o atributo protegido está associado ao target.
- **`feature_protected_correlation.csv`**: ajuda a identificar possíveis proxies do atributo protegido.
- **`feature_target_correlation.csv`**: mostra quais features estão mais associadas ao target.
- **`correlation_heatmap_top_features.png`**: resume as associações mais importantes em uma única figura.

Sugestão de uso no artigo: usar esses diagnósticos para justificar por que, em determinadas bases, pode haver perda maior de acurácia ao buscar maior equidade.